In [1]:
!pip install -q langchain-mistralai langgraph python-dotenv

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_mistralai import ChatMistralAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.graph.message import add_messages
from langgraph.store.base import BaseStore

In [3]:
from google.colab import userdata
import os
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

In [4]:
# DELIBERATELY hardcoded -- see MISTRAL_MODEL_LIMITS.md: mistral-small-latest
# has zero request allowance on many accounts (hard 429).
llm = ChatMistralAI(model="ministral-8b-latest")

In [6]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [8]:
def chat_node(state: ChatState, config, *, store: BaseStore):
    # store is injected AUTOMATICALLY by LangGraph when the graph is compiled
    # with a `store=` argument (see Cell 8) -- you never pass it yourself when
    # calling .invoke(); LangGraph wires it in for every node that declares a
    # `store: BaseStore` parameter.
    user_id = config["configurable"]["user_id"]

    # namespace groups related keys together, similar to a folder path -- here
    # scoped per-user, so different users' short-term memory never collides.
    namespace = ("memories", user_id)

    # store.search(...) looks up whatever was previously saved under this
    # namespace -- this is how earlier facts get pulled BACK into context.
    memories = store.search(namespace)
    info = "\n".join([d.value["data"] for d in memories])
    system_message = f"You are a helpful assistant. User info: {info}"

    messages = state["messages"]

    # Very simple rule: if the user says the word "remember", store the rest
    # of that message as a new memory. A real system would use an LLM to
    # decide WHAT is worth remembering rather than a keyword match, but this
    # keeps the mechanic itself easy to see.
    last_message = messages[-1]
    if "remember" in last_message.content.lower():
        memory = f"User said: {last_message.content}"
        # store.put(...) SAVES a new key/value under the namespace -- this is
        # the write side of the store, mirrored by store.search() as the read side.
        store.put(namespace, str(len(memories)), {"data": memory})

    response = llm.invoke([{"role": "system", "content": system_message}] + messages)

    return {"messages": [response]}

In [9]:
graph_builder = StateGraph(ChatState)

graph_builder.add_node("chat_node", chat_node)

graph_builder.add_edge(START, "chat_node")
graph_builder.add_edge("chat_node", END)

In [10]:
# checkpointer = conversation history (threads), same as your other chatbot labs.
# store = separate general-purpose key/value memory, NOT tied to any one thread --
# this is what lets a fact persist even across DIFFERENT conversation threads
# for the same user, unlike checkpointed messages which are scoped per-thread.
checkpointer = MemorySaver()
store = InMemoryStore()

graph = graph_builder.compile(checkpointer=checkpointer, store=store)

In [11]:
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

result = graph.invoke(
    {"messages": [HumanMessage(content="Please remember that I like the color blue.")]},
    config=config,
)

print(result["messages"][-1].content)

Got it! I’ll keep your love for **blue** in mind—whether it’s suggesting shades, designs, activities, or even just sprinkling in some blue-themed vibes. 😊

What’s on your mind today? Need inspiration, ideas, or just a little blue-themed fun? Let me know! 💙


In [12]:
result = graph.invoke(
    {"messages": [HumanMessage(content="What's 5 + 7?")]},
    config=config,
)

print(result["messages"][-1].content)

5 + 7 = **12**.

(And if you'd like, I could pair that with something blue—like a sky-blue background or a fun fact about the number 12, like how it’s the number of months in a year or the number of hours in a half-day! 😊)


In [13]:
config2 = {"configurable": {"thread_id": "2", "user_id": "1"}}

result = graph.invoke(
    {"messages": [HumanMessage(content="What color do I like?")]},
    config=config2,
)

print(result["messages"][-1].content)

I remember that you mentioned you like the color **blue**! 😊💙 Would you like suggestions for shades of blue or blue-themed ideas?
